In [ ]:
import sys
# Install missing integration packages along with core requirements
!{sys.executable} -m pip install --no-cache-dir --prefer-binary \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-chroma \
    langchain-text-splitters \
    sentence-transformers \
    chromadb \
    pypdf \
    tqdm \
    groq

In [1]:
import importlib
import sys
importlib.invalidate_caches()

try:
    import langchain_huggingface
    import langchain_chroma
    import langchain_text_splitters
    import torch
    print(f"Success: All integration libraries are available.")
    print(f"GPU available: {torch.cuda.is_available()}")
except ImportError as e:
    print(f"Import failed: {e}.")

Success: All integration libraries are available.
GPU available: True


In [2]:
import sys
try:
    import langchain_huggingface
    import langchain_chroma
    import torch
    print("✅ Environment is fully synced. You can now run the RAG pipeline cell (a7a858cf).")
except ImportError:
    print("❌ Packages still not found. Please ensure you have restarted the session via 'Runtime' -> 'Restart session'.")

✅ Environment is fully synced. You can now run the RAG pipeline cell (a7a858cf).


In [12]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from tqdm.auto import tqdm
import torch

# Configure Groq API
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.3-70b-versatile")

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load and Process Document
file_path = "/content/Seerat e Mustafa_new.pdf"
if not os.path.exists(file_path):
    print(f"Error: File {file_path} not found.")
    vectorstore = None
else:
    print("Loading and indexing PDF with real metadata...")
    loader = PyPDFLoader(file_path)
    pages = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    docs = text_splitter.split_documents(pages)

    # Use meaningful metadata: Page numbers and Source Title
    for i, doc in enumerate(docs):
        doc.metadata["chunk_id"] = i
        # Ensure we keep the actual page number from the loader
        doc.metadata["page"] = doc.metadata.get("page", 0)
        doc.metadata["source"] = "Seerat e Mustafa"

    hf_embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )

    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=hf_embeddings,
        collection_name="advanced_rag_eval"
    )
    print("✅ Vector store ready with real metadata.")

Loading and indexing PDF with real metadata...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store ready with real metadata.


In [19]:
import time
from langchain_groq import ChatGroq

# Using a smaller model for evaluation to stay within token limits
eval_llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama3-8b-8192")
rag.llm = eval_llm

if 'vectorstore' in globals() and vectorstore is not None:
    final_comparison_query = "Describe the main events of the Battle of Uhud, including the strategy of the archers and the turning point of the battle."

    print("--- GLOBAL SEARCH COMPARISON (Full PDF Indexed) ---")

    # 1. Baseline
    try:
        print("\n[1] Baseline Global Search:")
        b_docs = vectorstore.similarity_search(final_comparison_query, k=6)
        print(rag.get_answer(final_comparison_query, b_docs))
    except Exception as e:
        print(f"Error in Baseline: {e}")

    # 2. HyDE
    try:
        print("\n[2] HyDE Global Search:")
        h_docs = rag.hyde_retrieval(final_comparison_query)
        print(rag.get_answer(final_comparison_query, h_docs))
    except Exception as e:
        print(f"Error in HyDE: {e}")

    # 3. Sub-query
    try:
        print("\n[3] Sub-query Global Search:")
        sub_prompt = "Break this down into 2 specific sub-questions for: " + final_comparison_query
        sub_queries_resp = eval_llm.invoke(sub_prompt).content
        sub_queries = [q.strip() for q in sub_queries_resp.split('\n') if q.strip() and ('?' in q or q[0].isdigit())][:2]

        s_docs = []
        for sq in sub_queries:
            print(f"Searching for sub-query: {sq}")
            s_docs.extend(vectorstore.similarity_search(sq, k=3))
            time.sleep(5)

        print(rag.get_answer(final_comparison_query, s_docs))
    except Exception as e:
        print(f"Error in Sub-query: {e}")

else:
    print("Please ensure you have run the initialization cell (a7a858cf).")

--- GLOBAL SEARCH COMPARISON (Full PDF Indexed) ---

[1] Baseline Global Search:
Error in Baseline: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Waiting 10 seconds to respect rate limits...

[2] HyDE Global Search:
Error in HyDE: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Waiting 10 seconds to respect rate limits...

[3] Sub-query Global Search:
Error in Sub-query: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Pl